In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
RANDOM_STATE = 42
N_CLUSTERS = 4
TEST_SIZE = 0.2
SAFETY_BUFFER = 1.2
TOP_STATIONS = 8
kaggle_input = Path('/kaggle/input')
if kaggle_input.exists():
    csv_files = list(kaggle_input.rglob('a.csv'))
    DATASET_PATH = csv_files[0] if csv_files else None
else:
    DATASET_PATH = Path('data/a.csv')
if DATASET_PATH is None or not DATASET_PATH.exists():
    raise FileNotFoundError('Add a.csv as a Kaggle Notebook input, or save it locally as data/a.csv.')
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
ARTIFACT_PATH = OUTPUT_DIR / 'artifacts' / 'bike_demand_model.pkl'
print('Dataset:', DATASET_PATH)
print('Model will be saved to:', ARTIFACT_PATH)


In [ ]:
required_columns = ['start_time', 'duration', 'vehicle_type', 'start_station_id', 'start_station_name', 'temperature', 'max_temperature', 'min_temperature', 'wind_speed', 'precipitation', 'humidity', 'weather', 'sunshine', 'season', 'is_holiday']
raw = pd.read_csv(DATASET_PATH, usecols=required_columns, parse_dates=['start_time'], low_memory=False)
raw = raw.dropna(subset=['start_time', 'vehicle_type']).copy()
raw['hour_start'] = raw['start_time'].dt.floor('h')
raw['service_date'] = raw['start_time'].dt.normalize()
print('Rows:', len(raw))
print('Date range:', raw['start_time'].min(), 'to', raw['start_time'].max())
display(raw.head())
display(raw['vehicle_type'].value_counts())


In [ ]:
BIKE_TYPES = ['classic_bike', 'electric_bike']
WEATHER_COLUMNS = ['temperature', 'max_temperature', 'min_temperature', 'wind_speed', 'precipitation', 'humidity', 'sunshine']
bike_rows = raw[raw['vehicle_type'].isin(BIKE_TYPES)].copy()
if bike_rows.empty:
    raise ValueError('No classic_bike or electric_bike trips were found.')
bike_demand = bike_rows.groupby('hour_start').size().rename('bike_demand')
all_hours = pd.date_range(raw['hour_start'].min(), raw['hour_start'].max(), freq='h', name='hour_start')
hourly = pd.DataFrame(index=all_hours)
hourly['bike_demand'] = bike_demand.reindex(all_hours, fill_value=0)
hourly_weather = raw.groupby('hour_start')[WEATHER_COLUMNS].median()
daily_weather = raw.groupby('service_date')[WEATHER_COLUMNS].median()
for column in WEATHER_COLUMNS:
    values = hourly_weather[column].reindex(all_hours)
    daily_values = pd.Series([daily_weather.loc[time.normalize(), column] for time in all_hours], index=all_hours)
    hourly[column] = values.fillna(daily_values)
    hourly[column] = hourly[column].interpolate(method='time').ffill().bfill()
hourly_weather_code = raw.groupby('hour_start')['weather'].agg(lambda x: x.mode().iloc[0])
daily_weather_code = raw.groupby('service_date')['weather'].agg(lambda x: x.mode().iloc[0])
weather_codes = hourly_weather_code.reindex(all_hours)
daily_codes = pd.Series([daily_weather_code.loc[time.normalize()] for time in all_hours], index=all_hours)
hourly['weather_code'] = weather_codes.fillna(daily_codes).ffill().bfill().astype(int)
daily_season = raw.groupby('service_date')['season'].agg(lambda x: x.mode().iloc[0])
daily_holiday = raw.groupby('service_date')['is_holiday'].max()
hourly['season'] = [int(daily_season.loc[t.normalize()]) for t in all_hours]
hourly['is_holiday'] = [int(daily_holiday.loc[t.normalize()]) for t in all_hours]
print('Hourly rows:', len(hourly))
print('Zero-demand hours:', int((hourly['bike_demand'] == 0).sum()))
print('Bicycle trips counted:', int(hourly['bike_demand'].sum()))
display(hourly.head())


In [ ]:
hourly['hour'] = hourly.index.hour
hourly['day_of_week'] = hourly.index.dayofweek
hourly['month'] = hourly.index.month
hourly['day_of_year'] = hourly.index.dayofyear
hourly['is_weekend'] = (hourly['day_of_week'] >= 5).astype(int)
hourly['hour_sin'] = np.sin(2 * np.pi * hourly['hour'] / 24)
hourly['hour_cos'] = np.cos(2 * np.pi * hourly['hour'] / 24)
hourly['dow_sin'] = np.sin(2 * np.pi * hourly['day_of_week'] / 7)
hourly['dow_cos'] = np.cos(2 * np.pi * hourly['day_of_week'] / 7)
hourly['month_sin'] = np.sin(2 * np.pi * (hourly['month'] - 1) / 12)
hourly['month_cos'] = np.cos(2 * np.pi * (hourly['month'] - 1) / 12)
hourly['day_of_year_sin'] = np.sin(2 * np.pi * (hourly['day_of_year'] - 1) / 365.25)
hourly['day_of_year_cos'] = np.cos(2 * np.pi * (hourly['day_of_year'] - 1) / 365.25)
hourly['weather_code'] = hourly['weather_code'].astype(str)
hourly['season'] = hourly['season'].astype(str)
print('Final hourly dataset shape:', hourly.shape)
display(hourly[['bike_demand', 'hour', 'day_of_week', 'temperature', 'weather_code', 'season']].head())


In [ ]:
display(hourly['bike_demand'].describe())
hourly['bike_demand'].hist(bins=35, figsize=(8, 4))
plt.title('Hourly bicycle demand')
plt.xlabel('Bicycle trips in one hour')
plt.ylabel('Number of hours')
plt.show()
hourly.groupby('hour')['bike_demand'].mean().plot(kind='bar', figsize=(10, 4))
plt.title('Average bicycle demand by hour')
plt.xlabel('Hour of day')
plt.ylabel('Average trips')
plt.show()


In [ ]:
split_point = int(len(hourly) * (1 - TEST_SIZE))
split_time = hourly.index[split_point]
train_df = hourly.iloc[:split_point].copy()
test_df = hourly.iloc[split_point:].copy()
print('Training period:', train_df.index.min(), 'to', train_df.index.max())
print('Testing period: ', test_df.index.min(), 'to', test_df.index.max())
print('Training rows:', len(train_df))
print('Testing rows:', len(test_df))


In [ ]:
CLUSTER_NUMERIC_FEATURES = ['temperature', 'max_temperature', 'min_temperature', 'wind_speed', 'precipitation', 'humidity', 'sunshine', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos', 'is_holiday', 'is_weekend']
CLUSTER_CATEGORICAL_FEATURES = ['weather_code', 'season']
cluster_columns = CLUSTER_NUMERIC_FEATURES + CLUSTER_CATEGORICAL_FEATURES
cluster_preprocessor = ColumnTransformer([('numbers', StandardScaler(), CLUSTER_NUMERIC_FEATURES), ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CLUSTER_CATEGORICAL_FEATURES)])
X_train_cluster = cluster_preprocessor.fit_transform(train_df[cluster_columns])
X_test_cluster = cluster_preprocessor.transform(test_df[cluster_columns])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
train_cluster_ids = kmeans.fit_predict(X_train_cluster)
test_cluster_ids = kmeans.predict(X_test_cluster)
cluster_means = pd.Series(train_df['bike_demand'].values, index=train_cluster_ids).groupby(level=0).mean()
sorted_cluster_ids = cluster_means.sort_values().index.tolist()
profile_names = [f'Contextual demand profile {i}' for i in range(1, N_CLUSTERS + 1)]
cluster_label_map = {int(cluster_id): profile_names[i] for i, cluster_id in enumerate(sorted_cluster_ids)}
train_df['demand_profile'] = [cluster_label_map[int(x)] for x in train_cluster_ids]
test_df['demand_profile'] = [cluster_label_map[int(x)] for x in test_cluster_ids]
print('K-Means clustering complete.')
display(train_df.groupby('demand_profile')['bike_demand'].agg(['count', 'mean', 'median', 'std']))


In [ ]:
cluster_detail_stats = {}
for cluster_id in sorted_cluster_ids:
    profile_name = cluster_label_map[int(cluster_id)]
    values = train_df.loc[train_df['demand_profile'] == profile_name, 'bike_demand']
    cluster_detail_stats[profile_name] = {'cluster_id': int(cluster_id), 'count': int(values.count()), 'pct_of_training': round(float(values.count() / len(train_df) * 100), 1), 'mean': round(float(values.mean()), 2), 'median': round(float(values.median()), 1), 'std': round(float(values.std(ddof=0)), 2), 'min': int(values.min()), 'max': int(values.max()), 'p10': round(float(values.quantile(0.1)), 1), 'p25': round(float(values.quantile(0.25)), 1), 'p75': round(float(values.quantile(0.75)), 1), 'p90': round(float(values.quantile(0.9)), 1)}
cluster_summary = pd.DataFrame(cluster_detail_stats).T
display(cluster_summary)


In [ ]:
MODEL_NUMERIC_FEATURES = CLUSTER_NUMERIC_FEATURES.copy()
MODEL_CATEGORICAL_FEATURES = ['weather_code', 'season', 'demand_profile']
MODEL_FEATURES = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
model_preprocessor = ColumnTransformer([('numbers', 'passthrough', MODEL_NUMERIC_FEATURES), ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), MODEL_CATEGORICAL_FEATURES)])
model = Pipeline([('preprocessor', model_preprocessor), ('regressor', HistGradientBoostingRegressor(learning_rate=0.06, max_iter=300, max_leaf_nodes=20, min_samples_leaf=25, l2_regularization=2.0, early_stopping=True, random_state=RANDOM_STATE))])
model.fit(train_df[MODEL_FEATURES], train_df['bike_demand'])
model_predictions = np.maximum(0, model.predict(test_df[MODEL_FEATURES]))
print('Regression model training complete.')


In [ ]:
baseline_by_dow_hour = train_df.groupby(['day_of_week', 'hour'])['bike_demand'].mean().to_dict()
overall_training_mean = float(train_df['bike_demand'].mean())
baseline_predictions = np.array([baseline_by_dow_hour.get((int(row.day_of_week), int(row.hour)), overall_training_mean) for row in test_df.itertuples()])
actual = test_df['bike_demand'].to_numpy()

def calculate_metrics(actual_values, predicted_values):
    return {'MAE': mean_absolute_error(actual_values, predicted_values), 'RMSE': np.sqrt(mean_squared_error(actual_values, predicted_values)), 'R2': r2_score(actual_values, predicted_values)}
results = pd.DataFrame({'ML model': calculate_metrics(actual, model_predictions), 'Weekday-hour baseline': calculate_metrics(actual, baseline_predictions)}).T
display(results.round(3))
print('Lower MAE and RMSE are better. Higher R2 is generally better.')


In [ ]:
comparison = pd.DataFrame({'Actual': actual, 'ML model': model_predictions, 'Baseline': baseline_predictions}, index=test_df.index)
first_week_end = comparison.index.min() + pd.Timedelta(days=7)
comparison.loc[:first_week_end].plot(figsize=(13, 5), title='Actual demand vs model and baseline')
plt.ylabel('Bicycle trips per hour')
plt.xlabel('Time')
plt.show()


In [ ]:
train_bike_rows = bike_rows[bike_rows['hour_start'] < split_time].copy()
mean_trip_duration_minutes = float(train_bike_rows['duration'].clip(lower=0).mean() / 60)
median_trip_duration_minutes = float(train_bike_rows['duration'].clip(lower=0).median() / 60)
train_bike_rows['day_of_week'] = train_bike_rows['hour_start'].dt.dayofweek
train_bike_rows['hour'] = train_bike_rows['hour_start'].dt.hour
station_names = train_bike_rows.groupby(['start_station_id', 'start_station_name']).size().reset_index(name='trips').sort_values('trips', ascending=False).drop_duplicates('start_station_id')
station_name_lookup = dict(zip(station_names['start_station_id'], station_names['start_station_name']))

def get_station_shares(group):
    counts = group['start_station_id'].value_counts().head(TOP_STATIONS)
    total = group['start_station_id'].count()
    return [{'station_id': int(station_id), 'station_name': str(station_name_lookup.get(station_id, f'Station {station_id}')), 'share': float(count / total)} for station_id, count in counts.items()]
station_share_lookup = {}
for (day, hour), group in train_bike_rows.groupby(['day_of_week', 'hour']):
    station_share_lookup[int(day), int(hour)] = get_station_shares(group)
global_station_shares = get_station_shares(train_bike_rows)
station_count = int(train_bike_rows['start_station_id'].nunique())
print('Mean trip duration:', round(mean_trip_duration_minutes, 2), 'minutes')
print('Stations in training data:', station_count)
display(pd.DataFrame(station_share_lookup.get((0, 8), global_station_shares)))


In [ ]:
holiday_lookup = {str(date.date()): int(value) for date, value in daily_holiday.items()}
known_weather_codes = sorted((int(value) for value in hourly['weather_code'].unique()))
known_season_codes = sorted((int(value) for value in hourly['season'].unique()))
cluster_demand_means = {int(cluster_id): float(mean) for cluster_id, mean in cluster_means.items()}
artifact = {'artifact_version': 2, 'project_name': 'Smart Mobility Demand Forecasting and Capacity Allocation', 'target_name': 'hourly_bicycle_trip_starts', 'bike_vehicle_types': sorted(BIKE_TYPES), 'model': model, 'cluster_preprocessor': cluster_preprocessor, 'kmeans': kmeans, 'cluster_label_map': cluster_label_map, 'cluster_demand_means': cluster_demand_means, 'cluster_detail_stats': cluster_detail_stats, 'cluster_label_style': 'contextual', 'baseline_by_dow_hour': baseline_by_dow_hour, 'overall_training_mean': overall_training_mean, 'mean_trip_duration_minutes': mean_trip_duration_minutes, 'median_trip_duration_minutes': median_trip_duration_minutes, 'safety_buffer': SAFETY_BUFFER, 'station_share_lookup': station_share_lookup, 'global_station_shares': global_station_shares, 'station_count': station_count, 'model_numeric_features': MODEL_NUMERIC_FEATURES, 'model_categorical_features': MODEL_CATEGORICAL_FEATURES, 'cluster_numeric_features': CLUSTER_NUMERIC_FEATURES, 'cluster_categorical_features': CLUSTER_CATEGORICAL_FEATURES, 'training_start': str(train_df.index.min()), 'training_end': str(train_df.index.max()), 'dataset_start': str(hourly.index.min()), 'dataset_end': str(hourly.index.max()), 'test_start': str(test_df.index.min()), 'test_end': str(test_df.index.max()), 'metrics': results.to_dict(orient='index'), 'source_dataset_name': DATASET_PATH.name, 'holiday_lookup': holiday_lookup, 'known_weather_codes': known_weather_codes, 'known_season_codes': known_season_codes}
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, ARTIFACT_PATH)
print('Model bundle saved to:', ARTIFACT_PATH)
print('File size:', round(ARTIFACT_PATH.stat().st_size / 1024, 1), 'KB')


In [ ]:
loaded_artifact = joblib.load(ARTIFACT_PATH)
sample_features = test_df[MODEL_FEATURES].iloc[[0]]
sample_prediction = max(0, float(loaded_artifact['model'].predict(sample_features)[0]))
sample_time = test_df.index[0]
sample_profile = test_df['demand_profile'].iloc[0]
actual_demand = int(test_df['bike_demand'].iloc[0])
print('Sample time:', sample_time)
print('Contextual profile:', sample_profile)
print('Predicted trips:', round(sample_prediction, 2))
print('Actual trips:', actual_demand)
print('\nNotebook completed successfully.')
